# Análise das Transcrições das Lives do Frei Gilson - Quaresma 2025

## Introdução

**Autor:** *Bruno Conterato*

**Objetivo:** *Analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025 utilizando técnicas modernas de processamento de linguagem natural (NLP).*

---

## Descrição
Este notebook tem como objetivo analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025.

---

## Etapas de Pré-processamento
Este notebook descreve as etapas de pré-processamento necessárias para analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025.

---

## Necessidades de Processamento
- **Identificar o ponto de divisão** entre as reflexões do Terço e as reflexões do Dia.
- **Separar os textos** em reflexões do Terço e reflexões do Dia.
- **Remover as seções** marcadas com a tag `[Música]`.
- **Identificar e remover orações oficiais** da Igreja Católica.
- **Extrair e documentar músicas cantadas**, incluindo o nome e o autor de cada música.
- **Considerar o momento do Rosário** em que cada ensinamento foi apresentado, relacionando-o ao terço e ao mistério meditado naquele momento.

## 1. Configurações

### 1.1. Importação de Bibliotecas


In [ ]:
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.output_parsers.enum import EnumOutputParser, Enum
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers.pydantic import PydanticOutputParser
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field, field_validator
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain.output_parsers import RetryOutputParser
from langchain_core.documents import Document


from typing import List, Optional
import pandas as pd
import sqlite3
from tqdm.notebook import tqdm

from pprint import pprint

from dotenv import load_dotenv

load_dotenv()

In [ ]:
# MODEL = "deepseek-r1:8b"
# MODEL = "deepseek-r1:14b"
# MODEL = "deepseek-r1:8b-llama-distill-q4_K_M"
# MODEL = "deepseek-r1:8b-llama-distill-q8_0" # Trouxe só 1 versículo, e em Inglês
# MODEL = "deepseek-r1:14b-qwen-distill-q4_K_M" # Resumo ficou interessante. Traz versículos bíblicos.
# MODEL = "gemma3:12b"
# MODEL = "gemma3:27b-it-qat" # [18 GB] [Lento]
# MODEL = "gemma3:12b-it-q4_K_M"  # 8.1 GB
# MODEL = 'gemma3:12b-it-qat'   # [8.9GB]
# MODEL = "gemma3:12b-it-q8_0"  # [13 GB] # Identifica 2 versículos, 1 sem transcrição
# MODEL = "phi4:14b"
# MODEL = "mistral-small3.1:24b"
# MODEL = "DarkIdol-Llama-3.1-8B-Instruct-1.2-Uncensored:latest"
# MODEL = "llama3.1:8b"
# MODEL = "llama3.1:8b-instruct-q5_K_M"            # Bom resultado para versículos bíblicos
# MODEL = "llama3.1:8b-instruct-q8_0"            # Bom resultado para versículos bíblicos
# MODEL =  "llama3.1:8b-instruct-fp16"          # Bom resultado para versículos bíblicos
# MODEL = "mistral:7b"
# MODEL = "mistral:7b-instruct"
# MODEL = "openchat:7b"
# MODEL = "qwen2.5:7b-instruct"
# MODEL = "qwen3:8b"  # [5.2GB]    #Identificou corretamente 3 versículos bíblicos
# MODEL = "qwen3:14b" # [9.3GB]   # Identificou trechos de 2 versículos bíblicos
# MODEL = "nous-hermes2:10.7b"    # [6.1GB]
MODEL = "Gemma-3-Gaia-PT-BR-4b-it-f16:latest"

# List of Models: https://ai.google.dev/gemini-api/docs/models
# MODEL = "gemini-2.0-flash"

# Providers: azure_openai, deepseek, bedrock, openai, fireworks, xai, mistralai,
# ollama, google_anthropic_vertex, cohere, google_vertexai, perplexity,
# azure_ai, google_genai, ibm, bedrock_converse, groq, together, anthropic, huggingface
MODEL_PROVIDER = "ollama"
# MODEL_PROVIDER = "google_genai"

### 1.2. Hiperparâmetros

In [ ]:
MIN_SIMILARITY_THRESHOLD = 0.65
CHUNK_SIZE = 400
CHUNK_OVERLAP = 100

## 2.0. Processamento de texto

In [ ]:
def remove_starting_tabs(text):
    """
    Remove starting tabs from the text.
    """
    lines = text.split("\n")
    cleaned_lines = [line.lstrip("\t") for line in lines]
    return "\n".join(cleaned_lines)

In [ ]:
# system_message = f"""
#     Você é um especialista na fé católica.
#     Você receberá um texto que é uma transcrição de um vídeo do YouTube.

#     O vídeo é Santo Rosário, rezado diariamente ao vivo pelo Frei Gilson durante a um dia da quaresma do ano de 2025.
#     Durante o vídeo, o Frei Gilson reza o Santo Rosário e faz uma série de reflexões e ensinamentos sobre a fé católica.

#     O vídeo é dividido em duas partes bem distintas:
#     1. A primeira parte é o Santo Rosário, onde Frei Gilson reza o Rosário com os fiéis.
#     2. A segunda parte é uma reflexão relacionada à fé católica e ao silêncio.

#     Você vai se atentar a informações prestadas apenas durante a primeira parte do vídeo (o Santo Rosário),
#     e vai recuperar essas informações devidamente formatadas e organizadas,
#     a fim de que o usuário possa ter um resumo de todo ensinamento que foi transmitido.
    
#     Você vai trazer somente informações que estejam presentes na primeira parte do vídeo: o Santo Rosário.
#     Você não pode trazer informações que estejam presentes apenas na segunda parte do vídeo (a reflexão),
#     a menos que também o Santo Rosário do dia traga esse conhecimento.
    
#     Você não precisa explicar o funcionamento do Santo Rosário, 
#     a menos que essa informação seja relevante para algum ensinamento que você trouxer.
    
#     Você não precisa trazer transcricões de oracões católicas conhecidas, como o Pai Nosso, a Ave Maria ou o Credo.
    
#     Ignore as marcações de tempo presentes na transcrição.
#     Não invente informações, não faça suposições, não faça inferências, somente traga informações presentes na transcrição.
    
#     Você vai responder apenas com um relatório markdown, exatamente com as mesmas seções e formatações que estão no exemplo abaixo:
    
#     ```markdown
#     # Relatório do Santo Rosário
    
#     ## Temática principal
    
#     - <Neste Santo Rosário o Frei Gilson escolhe uma temática chave sobre a vida e a fé cristã, e a aborda com ensinamentos e reflexões entre as orações. Identifique a temática principal e traga-a aqui.>
#     <2 a 3 parágrafos de informações e ensinamentos sobre a temática principal dos ensinamentos trazidos junto a este Santo Rosário>
    
#     ## Temáticas secundárias
    
#     <O Frei Gilson também aborda temáticas secundárias entre as orações do Santo Rosário. Você vai identificá-las e trazer nas seções abaixo, começando pela mais relevante>
    
#     ### <temática secundária 1/2>
    
#     <1 a 2 parágrafos de informações e ensinamentos que foram transmitidos sobre a temática secundária 1>
    
#     ### <temática secundária 2/2>
    
#     <1 a 2 parágrafos de informações e ensinamentos que foram transmitidos sobre a temática secundária 2>
    
#     ...
    
#     ### <temática secundária N>
    
#     <1 a 2 parágrafos de informações e ensinamentos que foram transmitidos sobre a temática secundária N (No mínimo 2 e no máximo 5 temáticas secundárias)>
    
#     ## Versículos citados na transcrição
    
#     <Identifique cuidadosamente todos os versículos que foram citados na transmissão do Santo Rosário>
#     <Você vai trazer a localização do versículo, a transcrição integral do versículo e os ensinamentos que foram transmitidos por aquele versículo e pela narração do Frei Gilson>
#     <Se houver necessidade, identifique o momento do Santo Rosário em que o versículo foi citado (Qual terço, qual mistério, etc.)>
#     <Você não vai deixar de fora nenhum versículo que tenha sido citado>
#     <Pode haver intervalos de versículos, como por exemplo: Salmo 1, 1-3>
    
#     Ex.:
    
#     - Se o versículo Livro N, A (ou A-B caso seja m intervalo de versículos) foi citado, você vai trazer:
    
#     (Livro N, A) ou (Livro N, A-B): <Transcrição integral do versículo>
    
#     Ensinamentos: <Parágrafo de ensinamentos que foram transmitidos por aquele versículo>
    
#     ## Músicas
    
#     <Lista de todas as músicas que foram citadas durante o Santo Rosário>
    
#     Formato:
    
#     <Nome da música> - <Artista>
#     <Descrição do ensinamento que foi transmitido sobre a música>
    
#     ## Eventos de Agenda
    
#     <Lista de todos os eventos de agenda que foram citados durante o Santo Rosário>
    
#     Formato.:
    
#     <Título> - <data e horário> - <local>
#     <Descrição do evento>
#     ```
    
#     Por favor, não traga seções vazias, nem seções que não estejam no exemplo acima.
#     Nã responda nada além do relatório markdown exatamente como o exemplo acima.
# """

In [ ]:
# Aprimorado para LLMs menores pelo ChatGPT
system_message = """
    Você é um especialista na fé católica.  
    Receberá um **texto com a transcrição da oração do Santo Rosário**, rezado ao vivo pelo Frei Gilson durante um dia da Quaresma de 2025.

    A oração está dividida em duas partes:
    - A primeira parte é o Santo Rosário, onde Frei Gilson reza o Rosário com os fiéis.
    - A segunda parte é uma reflexão relacionada à fé católica e ao silêncio.

    ⚠️ **IMPORTANTE:**  
    Ignore qualquer parte da transcrição que seja da parte da **reflexão**.  
    Use **apenas o que for dito durante a oração do Rosário**.

    ---

    ### Sua missão é extrair e organizar as informações nos itens de 1 a 5 abaixo:

    ---

    ## 1. Temática principal

    - Identifique a principal ideia que Frei Gilson ensina enquanto reza o Rosário.
    - Resuma em até 3 parágrafos.
    - Use somente ensinamentos que ele falou **durante a oração do Santo Rosário**.

    ---

    ## 2. Temáticas secundárias

    - Identifique de 2 a 5 temas que Frei Gilson também falou **durante a oração do Santo ROsário**.
    - Para cada tema:
    - Dê um título claro (ex: *Perseverança na fé*).
    - Escreva 1 a 2 parágrafos com os ensinamentos relacionados.

    ---

    ## 3. Versículos da Bíblia

    Vou lhe fornecer os **versículos bíblicos citados durante a oração**.
    Você só ignorará versículos que sejam referentes a orações como o Pai nosso, Ave Maria, Credo etc,
    mas manterá todos os demais versículos.
    Não se esqueça de nenhum versículo ou intervalo de versículos que eu lhe fornecer, a menos que citado antyeriormente nas excessões.
    Traga apenas versículos que foram citados durante a oração do Santo Rosário.
    Traga a localização do versículo, a transcrição integral do versículo e os ensinamentos que foram transmitidos por aquele versículo e pela narração do Frei Gilson.
    Para cada um:

    - Escreva assim:  
    `(Livro Capítulo, Versículo)`: <Transcrição integral do versículo> ou
    `(Livro Capítulo, Versículo-Início–Versículo-Fim)`: <Transcrição integral do intervalo de versículos>

    - Logo abaixo, escreva:
    **Ensinamentos:**: <Parágrafo ou frase que explique o que o Frei falou sobre esse versículo durante a oração>

    Se o versículo for relacionado a algum mistério do Santo Rosário (como `Segundo Mistério Doloroso` ou `Quinto Mistério Gozozo`), diga isso.

    ---

    ## 4. Músicas

    Se Frei Gilson mencionar músicas durante a oração:

    - Diga o nome da música e o artista, como neste exemplo:
    <Nome da música> - <Artista>: <contexto>  

    - Depois escreva o que Frei Gilson disse sobre a música.

    ---

    ## 5. Eventos de Agenda

    Se Frei Gilson mencionar **eventos** (missas, encontros, lives etc.):
    Traga todos os eventos citados durante a oração ou ao final do Santo Rosário.
    - Traga o nome do evento, a data e o local.
    - Depois escreva o que ele falou sobre esse evento.

    ---

    ## ⚠️ Regras finais:

    - NÃO copie orações conhecidas (Pai Nosso, Ave Maria, Credo etc.).
    - Não explique o rosário, exceto se dor necessário para compreender o respectivo ensinamento.
    - NÃO traga informações da reflexão final que ocorre após o rosário. Traga só o que é dito durante o Rosário.
    - NÃO invente nada. Só use o que estiver escrito.
    - NÃO deixe seções em branco. Se algo não aparecer, **remova a seção inteira**.
    ```
"""

## 3.0 Detecção de trechos da bíblia

In [ ]:
# 0. Carregue a transcrição
transcription = """
Hoje refletimos sobre a importância de sermos humildes. Como está escrito: "Bem-aventurados os humildes, pois herdarão a terra".
Mais adiante, mencionou-se que devemos amar o próximo como a nós mesmos.
"""

# 1. Divida a transcrição
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.create_documents([transcription])

# 2. Carregue os embeddings e a base vetorial
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
db = Chroma(
    collection_name="biblia",
    persist_directory="../Bíblia VectorStore/biblia_vectorstore",
    embedding_function=embedding_model,
)


# 3. Defina a enum e o parser
class BinaryResponse(Enum):
    YES = "Sim."
    NO = "Não."


class SelectResponse(Enum):
    OPTION_1 = "1."
    OPTION_2 = "2."
    OPTION_3 = "3."


binary_parser = EnumOutputParser(enum=BinaryResponse)

select_parser = EnumOutputParser(enum=SelectResponse)


# 4. Modelo Pydantic para saída estruturada de referência bíblica
from utils import BookEnum, ORDERED_BOOKS


class BibleReference(BaseModel):
    book: BookEnum = Field(
        ...,
        description="Nome do livro da Bíblia explicitamente anunciado. Não deve conter número do capítulo.",
        examples=ORDERED_BOOKS,
    )
    chapter: int = Field(
        ...,
        description="Número do capítulo explicitamente anunciado. Não deve conter número de versículos.",
    )
    verse_start: int = Field(
        ...,
        description="Número do primeiro versículo do intervalo explicitamente anunciado, se for um único versículo. Se for um intervalo, é o primeiro versículo do intervalo.",
    )
    verse_end: int = Field(
        ...,
        description="Número do último versículo do intervalo explicitamente anunciado, se for um intervalo. Se não houver intervalo, igual ao verse_start.",
    )

    @field_validator("verse_end", mode="before")
    @classmethod
    def set_verse_end(cls, v, values):
        if v is None:
            return values.get("verse_start")
        return v


# Parser para saída estruturada
bible_reference_parser = JsonOutputParser(pydantic_object=BibleReference)

# Novo prompt: pede saída JSON estruturada para referência bíblica
bible_reference_prompt_template = """
Você é um assistente que analisa trechos de transcrição de fala e identifica qual é a **passagem bíblica** citada de forma clara e explícita.

Sua tarefa é identificar e **extrair** as seguintes informações, mesmo que existam variações de linguagem falada ou erros de transcrição:
- O **nome do livro da Bíblia**
- O **número do capítulo**
- O **número do(s) versículo(s)** — pode ser um único versículo (ex: 3) ou um intervalo (ex: 3-5)

Sua resposta deve ser exatamente neste formato **sem explicações, comentários ou qualquer outro texto**:
{format_instructions}

Texto a ser analisado:
{query}
"""

bible_reference_prompt = PromptTemplate(
    template=bible_reference_prompt_template,
    input_variables=["query"],
    partial_variables={
        "format_instructions": bible_reference_parser.get_format_instructions()
    },
)

print("Format Instructions:")
pprint(bible_reference_parser.get_format_instructions())
# 5. LLM local
# Funciona bem com os modelos instruct:
# - llama3.1:8b-instruct-q5_K_M
# - mistral:7b-instruct
llm = ChatOllama(model=MODEL)

# 6. Cadeia completa manual (RAG customizado)


def format_context(docs: List[Document], show_option=False):
    if show_option:
        return "\n\n".join(
            f"Excerpt {i+1}: {doc.metadata["book"]} {doc.metadata["chapter"]}:{doc.metadata["verse"]} {doc.page_content}"
            for (i, doc) in enumerate(docs)
        )
    return "\n\n".join(
        f"{doc.metadata["book"]} {doc.metadata["chapter"]}:{doc.metadata["verse"]} {doc.page_content}"
        for doc in docs
    )


def retrieve_similar_bible_passages(
    query: str, k: int = 3, min_similarity_threshould: float = MIN_SIMILARITY_THRESHOLD
):
    """
    Recupera as passagens bíblicas mais similares ao trecho fornecido.
    """
    retriever = db.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": min_similarity_threshould},
    )
    return retriever.invoke(query)[:k]


def retrieve_books_and_chapters(docs: List[Document]) -> str:
    """
    Retrieve all verses from the specific book and chapter found in docs.
    Assumes a table 'versiculos' with columns: id, book, chapter, verse, text, line_number.
    """
    conn = sqlite3.connect("../Bíblia VectorStore/biblia.db")
    cursor = conn.cursor()
    results = []
    seen = set()
    for doc in docs:
        book = doc.metadata.get("book")
        chapter = doc.metadata.get("chapter")
        if book and chapter and (book, chapter) not in seen:
            seen.add((book, chapter))
            results.append(f"\n\nLivro: {book} - Capítulo: {chapter}")
            cursor.execute(
                "SELECT book, chapter, verse, text FROM versiculos WHERE book=? AND chapter=? ORDER BY verse ASC",
                (book, chapter),
            )
            for row in cursor.fetchall():
                results.append(f"{row[2]}: {row[3]}")
    conn.close()
    return "\n".join(results) if results else "Nenhum resultado encontrado."

In [ ]:
# Pipeline binário para detecção de referência bíblica
binary_bible_reference_template = """
Você é um assistente que analisa trechos de transcrição de fala e identifica se há uma **citação bíblica clara e explícita**.

Sua tarefa é:
1. Verificar se o trecho contém uma **anunciação clara** de que será citada uma **passagem bíblica**.
2. A citação deve conter, de forma explícita (mesmo com erros de transcrição ou variações da fala):
   - O **nome do livro da Bíblia**
   - O **número do capítulo**
   - O **número do(s) versículo(s)** — pode ser um único versículo (ex: 3) ou um intervalo (ex: 3-5)

Se você encontrar essa citação bíblica clara, responda apenas com "Sim.". Caso contrário, responda apenas com "Não.".
Não escreva nenhuma explicação, justificativa ou comentário.
O resultado deve ser apenas "Sim." ou "Não." — nada além disso.

Texto a ser analisado:
{query}
"""

binary_bible_reference_prompt = PromptTemplate(
    template=binary_bible_reference_template,
    input_variables=["query"],
)

binary_bible_reference_chain = (
    {"query": lambda x: x["query"]}
    | binary_bible_reference_prompt
    | llm
    | binary_parser
)

In [ ]:
bible_reference_chain = (
    {"query": lambda x: x["query"]}
    | bible_reference_prompt
    | llm
    | bible_reference_parser
)

# retry_parser = RetryOutputParser.from_llm(parser=bible_reference_parser, llm=llm)

# bible_reference_chain = RunnableParallel(
#     completion=bible_reference_chain_untryed,
#     prompt_value=bible_reference_prompt,
# ) | RunnableLambda(lambda x: retry_parser.parse_with_prompt(**x))


def get_bible_passages(transcription: str, verbose: bool = False) -> list:
    """
    Extrai referências bíblicas de uma transcrição, com prints opcionais para depuração.
    """
    bible_passages = []
    chunks = splitter.create_documents([transcription])
    if verbose:
        print(f"[get_bible_passages] Total de chunks: {len(chunks)}")

    for idx, chunk in tqdm(
        enumerate(chunks), total=len(chunks), desc="Processando chunks"
    ):
        found_reference = False
        if verbose:
            print(f"\n[get_bible_passages] Chunk {idx+1}/{len(chunks)}:")
            print(f"Conteúdo do chunk:\n{chunk.page_content}")

        # 1. Primeiro, verifique se há referência bíblica usando o pipeline binário
        try:
            binary_result = binary_bible_reference_chain.invoke(
                {"query": chunk.page_content}
            )
            if verbose:
                print(f"Resultado do pipeline binário: {binary_result}")
            if binary_result == BinaryResponse.YES:
                found_reference = True
        except Exception as e:
            if verbose:
                print(f"[Aviso] Erro no pipeline binário: {e}")
            continue

        if not found_reference:
            if verbose:
                print("Nenhuma referência bíblica clara detectada neste chunk.")
            continue

        # 2. Se sim, execute o pipeline estruturado (sem retry)
        try:
            ref = bible_reference_chain.invoke({"query": chunk.page_content})
            if verbose:
                print(f"Referência bíblica extraída: {ref}")
            bible_passages.append(ref)
        except Exception as e:
            if verbose:
                print(f"[Aviso] Erro ao extrair referência estruturada: {e}")
            pass

    if verbose:
        print(
            f"\n[get_bible_passages] Total de referências extraídas: {len(bible_passages)}"
        )
    # Alternativa simples: usar pandas para remover duplicatas de dicionários

    # Se os itens forem dicionários ou BaseModel, converta para dicionário
    dicts = [
        passage.dict() if hasattr(passage, "dict") else passage
        for passage in bible_passages
    ]
    unique_passages = pd.DataFrame(dicts).drop_duplicates().to_dict(orient="records")
    return unique_passages


# Exemplo de uso:
bible_refs = get_bible_passages(
    transcription=remove_starting_tabs(
        """
        Livro de Primeiro João, capítulo 3, versículos 16 a 17.
        
        Não sabeis que sois o Templo de Deus, e que o Espírito de Deus habita em vós?
        Se alguém destruir o Templo de Deus, Deus o destruirá. 
        Porque o templo de Deus é sagrado – e isso sois vós.
    """
    ),
    verbose=True,
)
print("Referências bíblicas extraídas:")
for ref in bible_refs:
    print(ref)

## 4.0. Leitura dos Arquivos

In [ ]:
import os

from tqdm import tqdm
from langchain.chat_models import init_chat_model

model = init_chat_model(MODEL, model_provider=MODEL_PROVIDER)

raw_folder = "../../data/raw/Santo Rosário | Quaresma 2025/Youtube to Text"
processed_folder = "../../data/processed/Santo Rosário | Quaresma 2025/Youtube to Text"
titulo_template = (
    "Santo Rosário | Quaresma 2025 | 03:40 | {ordem}° Dia | Live Ao vivo.txt"
)

print("Diretório atual:", os.getcwd())


def gerar_titulo_fonte(ordem):
    return titulo_template.format(ordem=ordem)

for i in tqdm(range(13, 14), desc="Processando arquivos"):
    titulo_source = gerar_titulo_fonte(str(i))
    arquivo = f"{raw_folder}/{titulo_source}"
    titulo_md = f"{titulo_source[:-3]}.md"

    if os.path.exists(arquivo):
        with open(arquivo, "r+", encoding="utf-8") as f:
            conteudo = f.read()
            if conteudo:
                # TODO: retirar limites
                conteudo = conteudo[:20000]  # Limitar a 20.000 caracteres inicialmente
                print(f"\n\nArquivo: {titulo_source}")
                
                bible_refs = get_bible_passages(conteudo, verbose=True)
                print("Referências bíblicas extraídas:")
                for ref in bible_refs:
                    print(ref)
                
                # Versão antiga (com erro):
                # conteudo += "\n\nInstruções:\n\n" + remove_starting_tabs(system_message)
                # messages = [
                #     # {"role": "system", "content": remove_starting_tabs(system_message)},
                #     {"role": "user", "content": conteudo},
                # ]

                # Versão corrigida: system_message como mensagem de sistema
                messages = [
                    {"role": "system", "content": remove_starting_tabs(system_message)},
                    {"role": "user", "content": conteudo},
                ]
                    
                # response = model.invoke(messages).content
                
                chunks = []
                for chunk in model.stream(messages):
                    chunks.append(chunk.text())
                    print(chunk.text(), end="", flush=True)
                response = "".join(chunks)
                
                
                # with open(
                #     f"{processed_folder}/rosario_{titulo_md}", "w+", encoding="utf-8"
                # ) as f:
                #     f.write(response)

            else:
                print(f"\n\nO arquivo {titulo_source} está vazio.")
    else:
        print(f"\n\nArquivo não encontrado: {arquivo}")
    
    break